## Make Detection with the Trained Model

In [1]:
import mediapipe as mp
import cv2
import numpy as np
import pandas as pd

import pickle

import warnings
warnings.filterwarnings('ignore')

# Drawing helpers
mp_drawing = mp.solutions.drawing_utils
mp_pose = mp.solutions.pose

### Reconstruct the input structure

In [2]:
# Determine important landmarks for plank
IMPORTANT_LMS = [
    "NOSE",
    "LEFT_SHOULDER",
    "RIGHT_SHOULDER",
    "LEFT_ELBOW",
    "RIGHT_ELBOW",
    "LEFT_WRIST",
    "RIGHT_WRIST",
    "LEFT_HIP",
    "RIGHT_HIP",
    "LEFT_KNEE",
    "RIGHT_KNEE",
    "LEFT_ANKLE",
    "RIGHT_ANKLE",
    "LEFT_HEEL",
    "RIGHT_HEEL",
    "LEFT_FOOT_INDEX",
    "RIGHT_FOOT_INDEX",
]

# Generate all columns of the data frame

HEADERS = ["label"] # Label column

for lm in IMPORTANT_LMS:
    HEADERS += [f"{lm.lower()}_x", f"{lm.lower()}_y", f"{lm.lower()}_z", f"{lm.lower()}_v"]

### Setup some important functions

In [3]:
def extract_important_keypoints(results) -> list:
    '''
    Extract important keypoints from mediapipe pose detection
    '''
    landmarks = results.pose_landmarks.landmark

    data = []
    for lm in IMPORTANT_LMS:
        keypoint = landmarks[mp_pose.PoseLandmark[lm].value]
        data.append([keypoint.x, keypoint.y, keypoint.z, keypoint.visibility])
    
    return np.array(data).flatten().tolist()


def rescale_frame(frame, percent=50):
    '''
    Rescale a frame to a certain percentage compare to its original frame
    '''
    width = int(frame.shape[1] * percent/ 100)
    height = int(frame.shape[0] * percent/ 100)
    dim = (width, height)
    return cv2.resize(frame, dim, interpolation =cv2.INTER_AREA)


# def rescale_frame_to_max(frame, max_dim=800):
#     """
#     Resize so the largest side is max_dim (keeps aspect ratio).
#     Returns original if already smaller.
#     """
#     h, w = frame.shape[:2]
#     if max(h, w) <= max_dim:
#         return frame
#     scale = max_dim / max(h, w)
#     dim = (int(w * scale), int(h * scale))
#     return cv2.resize(frame, dim, interpolation=cv2.INTER_AREA)


In [4]:
# VIDEO_TEST = "../../demo/plank_demo.mp4"
# VIDEO_TEST = "Phalakasana-2.mp4"
# VIDEO_TEST = "Phalakasana-9.mp4"
VIDEO_TEST = "plank_demo.mp4"
# VIDEO_TEST = "plank test.mp4"

## 1. Make detection with Scikit learn model

In [ ]:
# Load model
# with open("./model/LR_model.pkl", "rb") as f:
#     sklearn_model = pickle.load(f)



# with open("./model/xgboost_model.pkl", "rb") as f:
#     model = pickle.load(f)

# # Load input scaler
# with open("./model/input_scaler.pkl", "rb") as f2:
#     input_scaler = pickle.load(f2)


with open("./model/plank_pipeline.pkl", "rb") as f:
    plank_pipeline = pickle.load(f)

# Transform prediction into class
def get_class(prediction: float) -> str:
    return {
        0: "C",
        1: "H",
        2: "L",
    }.get(prediction)


In [6]:
import sys
import os

# Add the path to the parent folder of 'angle_calculation'
sys.path.append(os.path.abspath(os.path.join(os.path.dirname(''), '../angle_calculation')))


In [7]:
from plank_angle_calculation import compute_shoulder_angle, compute_elbow_angle, compute_hip_angle, compute_knee_angle
def compute_all_angles(row):
    """
    Compute all required angles for plank pose.
    Returns a list of angles in the same order used during training.
    """
    angles = [
        compute_shoulder_angle(row, side="left"),
        compute_elbow_angle(row, side="left"),
        compute_hip_angle(row, side="left"),
        compute_knee_angle(row, side="left"),
        compute_shoulder_angle(row, side="right"),
        compute_elbow_angle(row, side="right"),
        compute_hip_angle(row, side="right"),
        compute_knee_angle(row, side="right"),
    ]
    return angles


In [8]:
# Angle column names (same order as compute_all_angles)
ANGLE_COLUMNS = [
    "left_shoulder_angle",   # shoulder - hip - elbow
    "left_elbow_angle",
    "left_hip_angle",        # shoulder - hip - knee
    "left_knee_angle",       # hip - knee - ankle
    "right_shoulder_angle",      # shoulder - elbow - wrist
    "right_elbow_angle",
    "right_hip_angle",
    "right_knee_angle"
]


In [9]:
for i, label in enumerate(plank_pipeline.named_steps['model'].classes_):
    print(f"{i} → {label}")

0 → 0
1 → 1
2 → 2


In [10]:
cap = cv2.VideoCapture(VIDEO_TEST)
current_stage = ""
prediction_probability_threshold = 0.8

with mp_pose.Pose(min_detection_confidence=0.5, min_tracking_confidence=0.5) as pose:
    while cap.isOpened():
        ret, image = cap.read()

        if not ret:
            break

        # Reduce size of a frame
        image = rescale_frame(image, 50)
        # image = cv2.flip(image, 1)

        # Recolor image from BGR to RGB for mediapipe
        image = cv2.cvtColor(image, cv2.COLOR_BGR2RGB)
        image.flags.writeable = False

        results = pose.process(image)

        if not results.pose_landmarks:
            print("No human found")
            continue

        # Recolor image from BGR to RGB for mediapipe
        image.flags.writeable = True
        image = cv2.cvtColor(image, cv2.COLOR_RGB2BGR)

        # Draw landmarks and connections
        mp_drawing.draw_landmarks(image, results.pose_landmarks, mp_pose.POSE_CONNECTIONS, mp_drawing.DrawingSpec(color=(244, 117, 66), thickness=2, circle_radius=2), mp_drawing.DrawingSpec(color=(245, 66, 230), thickness=2, circle_radius=1))

        # Make detection
        try:
            # Extract keypoints from frame for the input
            row = extract_important_keypoints(results)  # list of landmarks
            X_landmarks = pd.DataFrame([row], columns=HEADERS[1:])  # only landmarks columns

            # Compute all angles for this frame
            angles = compute_all_angles(X_landmarks.iloc[0])  # returns list of angles in the same order as training
            X_angles = pd.DataFrame([angles], columns=ANGLE_COLUMNS)  # ANGLE_COLUMNS = list of your angle column names

            # Combine landmarks + angles
            X_full = pd.concat([X_landmarks, X_angles], axis=1)

            # Predict with the pipeline (no need to scale manually)
            predicted_class = plank_pipeline.predict(X_full)[0]
            predicted_class = get_class(predicted_class)

            prediction_probability = plank_pipeline.predict_proba(X_full)[0]
            print(predicted_class, prediction_probability)

            # # Evaluate model prediction
            # if predicted_class == 0 and prediction_probability[prediction_probability.argmax()] >= prediction_probability_threshold:
            #     current_stage = "Correct"
            # elif predicted_class == 2 and prediction_probability[prediction_probability.argmax()] >= prediction_probability_threshold: 
            #     current_stage = "Low back"
            # elif predicted_class == 1 and prediction_probability[prediction_probability.argmax()] >= prediction_probability_threshold: 
            #     current_stage = "High back"
            # else:
            #     current_stage = "unk"
            
            if predicted_class == "C" and prediction_probability[prediction_probability.argmax()] >= prediction_probability_threshold:
                current_stage = "Correct"
            elif predicted_class == "L" and prediction_probability[prediction_probability.argmax()] >= prediction_probability_threshold: 
                current_stage = "Low back"
            elif predicted_class == "H" and prediction_probability[prediction_probability.argmax()] >= prediction_probability_threshold: 
                current_stage = "High back"
            else:
                current_stage = "unk"
            
            # Visualization
            # Status box
            cv2.rectangle(image, (0, 0), (250, 60), (245, 117, 16), -1)

            # Display class
            cv2.putText(image, "CLASS", (95, 12), cv2.FONT_HERSHEY_COMPLEX, 0.5, (0, 0, 0), 1, cv2.LINE_AA)
            cv2.putText(image, current_stage, (90, 40), cv2.FONT_HERSHEY_COMPLEX, 1, (255, 255, 255), 2, cv2.LINE_AA)

            # Display probability
            cv2.putText(image, "PROB", (15, 12), cv2.FONT_HERSHEY_COMPLEX, 0.5, (0, 0, 0), 1, cv2.LINE_AA)
            cv2.putText(image, str(round(prediction_probability[np.argmax(prediction_probability)], 2)), (10, 40), cv2.FONT_HERSHEY_COMPLEX, 1, (255, 255, 255), 2, cv2.LINE_AA)

        except Exception as e:
            print(f"Error: {e}")
        
        cv2.imshow("CV2", image)
        
        # Press Q to close cv2 window
        if cv2.waitKey(1) & 0xFF == ord('q'):
            break

    cap.release()
    cv2.destroyAllWindows()

    

    for i in range (1, 5):
        cv2.waitKey(1)
  

C [9.9995363e-01 9.3945309e-06 3.6995494e-05]
C [9.999528e-01 9.394523e-06 3.786744e-05]
C [9.9993885e-01 9.3943918e-06 5.1700510e-05]
C [9.9995589e-01 6.3918205e-06 3.7701873e-05]
C [9.9995589e-01 6.3918205e-06 3.7701873e-05]
C [9.9997032e-01 6.3919128e-06 2.3233522e-05]
C [9.999689e-01 6.707505e-06 2.438067e-05]
C [9.9997568e-01 6.7075503e-06 1.7581300e-05]
C [9.9994683e-01 1.1701754e-05 4.1483236e-05]
C [9.9990511e-01 1.4003834e-05 8.0915488e-05]
C [9.9981886e-01 1.5073843e-05 1.6608172e-04]
C [9.9973148e-01 1.8529377e-05 2.4999876e-04]
C [9.989813e-01 2.307831e-05 9.955569e-04]
C [9.9773043e-01 5.7205070e-05 2.2123742e-03]
C [9.9738687e-01 5.7185374e-05 2.5560034e-03]
C [9.9738687e-01 5.7185374e-05 2.5560034e-03]
C [9.9242330e-01 8.8925633e-05 7.4878037e-03]
C [9.8944968e-01 1.5427497e-04 1.0396063e-02]
C [9.8152703e-01 2.7012528e-04 1.8202819e-02]
C [9.8080516e-01 2.8068142e-04 1.8914150e-02]
C [9.6712863e-01 4.2102195e-04 3.2450344e-02]
C [9.6807456e-01 2.7703823e-04 3.1648379e-0

In [ ]:
# cap = cv2.VideoCapture(VIDEO_TEST)
# current_stage = ""
# prediction_probability_threshold = 0.82

# with mp_pose.Pose(min_detection_confidence=0.5, min_tracking_confidence=0.5) as pose:
#     while cap.isOpened():
#         ret, image = cap.read()

#         if not ret:
#             break

#         # Reduce size of a frame
#         image = rescale_frame(image, 50)
#         # image = cv2.flip(image, 1)

#         # Recolor image from BGR to RGB for mediapipe
#         image = cv2.cvtColor(image, cv2.COLOR_BGR2RGB)
#         image.flags.writeable = False

#         results = pose.process(image)

#         if not results.pose_landmarks:
#             print("No human found")
#             continue

#         # Recolor image from BGR to RGB for mediapipe
#         image.flags.writeable = True
#         image = cv2.cvtColor(image, cv2.COLOR_RGB2BGR)

#         # Draw landmarks and connections
#         mp_drawing.draw_landmarks(image, results.pose_landmarks, mp_pose.POSE_CONNECTIONS, mp_drawing.DrawingSpec(color=(244, 117, 66), thickness=2, circle_radius=2), mp_drawing.DrawingSpec(color=(245, 66, 230), thickness=2, circle_radius=1))

#         # Make detection
#         try:
#             # Extract keypoints from frame for the input
#             row = extract_important_keypoints(results)
#             X = pd.DataFrame([row], columns=HEADERS[1:])
#             # Scale features (keep as DataFrame)
#             X_scaled = pd.DataFrame(input_scaler.transform(X), columns=X.columns)

#             # Make prediction and its probability
#             predicted_class = sklearn_model.predict(X_scaled)[0]
#             predicted_class = get_class(predicted_class)
#             prediction_probability = sklearn_model.predict_proba(X_scaled)[0]
#             # print(predicted_class, prediction_probability)

#             # Evaluate model prediction
#             if predicted_class == "C" and prediction_probability[prediction_probability.argmax()] >= prediction_probability_threshold:
#                 current_stage = "Correct"
#             elif predicted_class == "L" and prediction_probability[prediction_probability.argmax()] >= prediction_probability_threshold: 
#                 current_stage = "Low back"
#             elif predicted_class == "H" and prediction_probability[prediction_probability.argmax()] >= prediction_probability_threshold: 
#                 current_stage = "High back"
#             else:
#                 current_stage = "unk"
            
#             # Visualization
#             # Status box
#             cv2.rectangle(image, (0, 0), (250, 60), (245, 117, 16), -1)

#             # Display class
#             cv2.putText(image, "CLASS", (95, 12), cv2.FONT_HERSHEY_COMPLEX, 0.5, (0, 0, 0), 1, cv2.LINE_AA)
#             cv2.putText(image, current_stage, (90, 40), cv2.FONT_HERSHEY_COMPLEX, 1, (255, 255, 255), 2, cv2.LINE_AA)

#             # Display probability
#             cv2.putText(image, "PROB", (15, 12), cv2.FONT_HERSHEY_COMPLEX, 0.5, (0, 0, 0), 1, cv2.LINE_AA)
#             cv2.putText(image, str(round(prediction_probability[np.argmax(prediction_probability)], 2)), (10, 40), cv2.FONT_HERSHEY_COMPLEX, 1, (255, 255, 255), 2, cv2.LINE_AA)

#         except Exception as e:
#             print(f"Error: {e}")
        
#         cv2.imshow("CV2", image)
        
#         # Press Q to close cv2 window
#         if cv2.waitKey(1) & 0xFF == ord('q'):
#             break

#     cap.release()
#     cv2.destroyAllWindows()

    

#     for i in range (1, 5):
#         cv2.waitKey(1)
  

In [ ]:
# cap = cv2.VideoCapture(VIDEO_TEST)
# current_stage = ""
# prediction_probability_threshold = 0.82

# with mp_pose.Pose(min_detection_confidence=0.5, min_tracking_confidence=0.5) as pose:
#     while cap.isOpened():
#         ret, image = cap.read()

#         if not ret:
#             break

#         # Reduce size of a frame
#         image = rescale_frame(image, 50)
#         # image = cv2.flip(image, 1)

#         # Recolor image from BGR to RGB for mediapipe
#         image = cv2.cvtColor(image, cv2.COLOR_BGR2RGB)
#         image.flags.writeable = False

#         results = pose.process(image)

#         if not results.pose_landmarks:
#             print("No human found")
#             continue

#         # Recolor image from BGR to RGB for mediapipe
#         image.flags.writeable = True
#         image = cv2.cvtColor(image, cv2.COLOR_RGB2BGR)

#         # Draw landmarks and connections
#         mp_drawing.draw_landmarks(image, results.pose_landmarks, mp_pose.POSE_CONNECTIONS, mp_drawing.DrawingSpec(color=(244, 117, 66), thickness=2, circle_radius=2), mp_drawing.DrawingSpec(color=(245, 66, 230), thickness=2, circle_radius=1))

#         # Make detection
#         try:
#             # Extract keypoints from frame for the input
#             row = extract_important_keypoints(results)
#             X = pd.DataFrame([row], columns=HEADERS[1:])
#             X = pd.DataFrame(input_scaler.transform(X))

#             # Make prediction and its probability
#             predicted_class = sklearn_model.predict(X)[0]
#             predicted_class = get_class(predicted_class)
#             prediction_probability = sklearn_model.predict_proba(X)[0]
#             # print(predicted_class, prediction_probability)

#             # Evaluate model prediction
#             if predicted_class == "C" and prediction_probability[prediction_probability.argmax()] >= prediction_probability_threshold:
#                 current_stage = "Correct"
#             elif predicted_class == "L" and prediction_probability[prediction_probability.argmax()] >= prediction_probability_threshold: 
#                 current_stage = "Low back"
#             elif predicted_class == "H" and prediction_probability[prediction_probability.argmax()] >= prediction_probability_threshold: 
#                 current_stage = "High back"
#             else:
#                 current_stage = "unk"
            
#             # Visualization
#             # Status box
#             cv2.rectangle(image, (0, 0), (250, 60), (245, 117, 16), -1)

#             # Display class
#             cv2.putText(image, "CLASS", (95, 12), cv2.FONT_HERSHEY_COMPLEX, 0.5, (0, 0, 0), 1, cv2.LINE_AA)
#             cv2.putText(image, current_stage, (90, 40), cv2.FONT_HERSHEY_COMPLEX, 1, (255, 255, 255), 2, cv2.LINE_AA)

#             # Display probability
#             cv2.putText(image, "PROB", (15, 12), cv2.FONT_HERSHEY_COMPLEX, 0.5, (0, 0, 0), 1, cv2.LINE_AA)
#             cv2.putText(image, str(round(prediction_probability[np.argmax(prediction_probability)], 2)), (10, 40), cv2.FONT_HERSHEY_COMPLEX, 1, (255, 255, 255), 2, cv2.LINE_AA)

#         except Exception as e:
#             print(f"Error: {e}")
        
#         cv2.imshow("CV2", image)
        
#         # Press Q to close cv2 window
#         if cv2.waitKey(1) & 0xFF == ord('q'):
#             break

#     cap.release()
#     cv2.destroyAllWindows()

    

#     for i in range (1, 5):
#         cv2.waitKey(1)
  

## 2. Make detection with Deep Learning Model

In [ ]:
# Load model
with open("./model/plank_dp.pkl", "rb") as f:
    deep_learning_model = pickle.load(f)

In [ ]:
cap = cv2.VideoCapture(VIDEO_TEST)
current_stage = ""
prediction_probability_threshold = 0.8

with mp_pose.Pose(min_detection_confidence=0.5, min_tracking_confidence=0.5) as pose:
    while cap.isOpened():
        ret, image = cap.read()

        if not ret:
            break

        # Reduce size of a frame
        image = rescale_frame(image, 50)

        # Recolor image from BGR to RGB for mediapipe
        image = cv2.cvtColor(image, cv2.COLOR_BGR2RGB)
        image.flags.writeable = False

        results = pose.process(image)

        if not results.pose_landmarks:
            print("No human found")
            continue

        # Recolor image from BGR to RGB for mediapipe
        image.flags.writeable = True
        image = cv2.cvtColor(image, cv2.COLOR_RGB2BGR)

        # Draw landmarks and connections
        mp_drawing.draw_landmarks(image, results.pose_landmarks, mp_pose.POSE_CONNECTIONS, mp_drawing.DrawingSpec(color=(244, 117, 66), thickness=2, circle_radius=2), mp_drawing.DrawingSpec(color=(245, 66, 230), thickness=2, circle_radius=1))

        # Make detection
        try:
            # Extract keypoints from frame for the input
            row = extract_important_keypoints(results)
            X = pd.DataFrame([row, ], columns=HEADERS[1:])
            X = pd.DataFrame(input_scaler.transform(X))
            

            # Make prediction and its probability
            prediction = deep_learning_model.predict(X)
            predicted_class = np.argmax(prediction, axis=1)[0]

            prediction_probability = max(prediction.tolist()[0])
            # print(X)

            # Evaluate model prediction
            if predicted_class == 0 and prediction_probability >= prediction_probability_threshold:
                current_stage = "Correct"
            elif predicted_class == 2 and prediction_probability >= prediction_probability_threshold: 
                current_stage = "Low back"
            elif predicted_class == 1 and prediction_probability >= prediction_probability_threshold: 
                current_stage = "High back"
            else:
                current_stage = "Unknown"

            # Visualization
            # Status box
            cv2.rectangle(image, (0, 0), (550, 60), (245, 117, 16), -1)

            # # Display class
            cv2.putText(image, "DETECTION", (95, 12), cv2.FONT_HERSHEY_COMPLEX, 0.5, (0, 0, 0), 1, cv2.LINE_AA)
            cv2.putText(image, current_stage, (90, 40), cv2.FONT_HERSHEY_COMPLEX, 1, (255, 255, 255), 2, cv2.LINE_AA)

            # # Display class
            cv2.putText(image, "CLASS", (350, 12), cv2.FONT_HERSHEY_COMPLEX, 0.5, (0, 0, 0), 1, cv2.LINE_AA)
            cv2.putText(image, str(predicted_class), (345, 40), cv2.FONT_HERSHEY_COMPLEX, 1, (255, 255, 255), 2, cv2.LINE_AA)

            # # Display probability
            cv2.putText(image, "PROB", (15, 12), cv2.FONT_HERSHEY_COMPLEX, 0.5, (0, 0, 0), 1, cv2.LINE_AA)
            cv2.putText(image, str(round(prediction_probability, 2)), (10, 40), cv2.FONT_HERSHEY_COMPLEX, 1, (255, 255, 255), 2, cv2.LINE_AA)

        except Exception as e:
            print(f"Error: {e}")
        
        cv2.imshow("CV2", image)
        
        # Press Q to close cv2 window
        if cv2.waitKey(1) & 0xFF == ord('q'):
            break

    cap.release()
    cv2.destroyAllWindows()

    for i in range (1, 5):
        cv2.waitKey(1)
  

In [ ]:
X